<a href="https://colab.research.google.com/github/JiyadR/BigData26_A_2411533003_Jiyad-Rifqi-Pasaribu/blob/main/Praktikum2/BD_A_P02_2411533003_Jiyad_Rifqi_Pasaribu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Perintah ini menginstal library Faker yang digunakan untuk membuat data palsu (seperti nama orang, alamat, tanggal) secara otomatis untuk keperluan testing atau simulasi dataset.

In [1]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 32.5 MB/s eta 0:00:00


### **K-1. Import Library dan Inisialisasi**

Mengimpor library yang dibutuhkan untuk membuat dan mengolah data. NumPy untuk operasi numerik, Pandas untuk data tabular, Faker untuk data palsu, dan random untuk nilai acak.

In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

### **K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)**

Kode ini membuat dataset transaksi marketplace palsu dengan 500 baris menggunakan SEED 42 agar hasilnya selalu sama. Data sengaja dibuat "kotor" dengan format harga dan tanggal yang berantakan, kapitalisasi tidak konsisten, sebagian data kosong (missing value sekitar 2-3%), dan 15 baris duplikat untuk mensimulasikan kondisi data nyata yang tidak sempurna. Hasil akhir disimpan ke file transaksi_mentah.csv dengan total 515 baris (500 asli + 15 duplikat).

In [3]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


### **K-3. Deteksi dan Penanganan Missing Value**

Kode ini mendeteksi jumlah data kosong di setiap kolom, hasilnya ada 20 nama pelanggan, 16 metode pembayaran, 30 kota, dan 166 rating yang kosong. Baris dengan nama pelanggan atau metode pembayaran kosong dihapus karena informasi ini wajib ada, sedangkan kota yang kosong diisi dengan teks "Tidak Diketahui" agar datanya tetap berguna untuk analisis lain.

In [4]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [5]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

### **K-4. Deteksi dan Penanganan Duplicate**

Kode ini mendeteksi baris yang benar-benar identik (duplicate) dan menemukan 5 baris duplikat. Setelah baris duplikat dihapus menggunakan drop_duplicates(), jumlah data berkurang dari 515 menjadi 490 baris, sehingga hanya menyisakan data transaksi yang unik saja.

In [6]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df["transaction_id"].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


### **K-5. Koreksi Tipe Data dan Standardisasi Format**

Kode ini merapikan seluruh format data agar konsisten dan siap dianalisis dengan empat langkah utama. Pertama, menyeragamkan teks kategorikal menjadi Title Case dan menghapus spasi berlebih, khusus "COD" dikembalikan ke huruf kapital semua. Kedua, membersihkan kolom harga dari simbol "Rp", titik ribuan, dan spasi lalu mengubahnya menjadi angka murni (float). Ketiga, menyeragamkan format tanggal yang berantakan (ISO, DD/MM/YYYY, DD-MM-YYYY) menjadi satu format standar YYYY-MM-DD dengan mencoba berbagai format secara berurutan hingga berhasil.

In [7]:
# a. Standardisasi teks kategorikal (category, payment_method, shipping_city):

for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})


# b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik):

def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)


# c. Standardisasi format tanggal ke YYYY-MM-DD:

def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")


# d. Finalisasi tipe data:

df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

### **K-6. Ekspor Dataset Bersih**

Kode ini menyimpan hasil akhir dataset yang sudah dibersihkan ke file transaksi_bersih.csv tanpa menyertakan nomor indeks. Dataset yang awalnya 515 baris (mentah) kini menjadi 490 baris setelah dihapus data kosong dan duplikat, siap digunakan untuk analisis atau praktikum berikutnya.

In [8]:
# K-6. Ekspor Dataset Bersih

df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


In [9]:
# Upload file CSV ke Google Drive

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Buat folder khusus (jika belum ada)
folder_path = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(folder_path, exist_ok=True)

# Copy file ke folder tersebut
!cp transaksi_mentah.csv "{folder_path}/"
!cp transaksi_bersih.csv "{folder_path}/"

print("✅ File berhasil diupload ke Google Drive!")
print(f"📁 Lokasi: MyDrive/Praktikum_BigData/")
print(f"📄 File yang diupload:")
print(f"   - transaksi_mentah.csv")
print(f"   - transaksi_bersih.csv")

Mounted at /content/drive
✅ File berhasil diupload ke Google Drive!
📁 Lokasi: MyDrive/Praktikum_BigData/
📄 File yang diupload:
   - transaksi_mentah.csv
   - transaksi_bersih.csv


### **Studi Kasus**

1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda lakukan.
Kedua angka tersebut berbeda karena selama proses pra-pemrosesan data, sebanyak 25 baris dihapus dari total 515 baris awal. Penghapusan ini terjadi dalam dua tahap: pertama, 20 baris dihapus karena memiliki missing value pada kolom yang bersifat wajib (customer_name dan payment_method) menggunakan fungsi dropna(), dan kedua, 5 baris dihapus karena merupakan data duplikat menggunakan fungsi drop_duplicates(). Jadi, 515 baris (data mentah) - 20 baris (missing value) - 5 baris (duplikat) = 490 baris (data bersih).

2. Apakah 490 baris "lebih benar" dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep Veracity.
Ya, 490 baris lebih benar dibanding 515 baris jika dilihat dari konsep Veracity (kebenaran/akurasi data) dalam 5V Big Data. Dataset dengan 490 baris memiliki Veracity yang lebih tinggi karena telah dibersihkan dari data yang tidak valid (missing value pada kolom wajib) dan data duplikat yang dapat menyebabkan hasil analisis menjadi bias atau salah. Meskipun Volume datanya berkurang, kualitas dan kepercayaan terhadap data tersebut meningkat, sehingga analisis yang dilakukan akan menghasilkan insight yang lebih akurat dan dapat diandalkan untuk pengambilan keputusan. Konsep "garbage in, garbage out" menjelaskan bahwa data berkualitas rendah akan menghasilkan hasil analisis yang salah, sehingga 490 baris data bersih lebih bernilai daripada 515 baris data kotor.

3. Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing value kepada tim Finance yang ingin tahu "rating rata-rata semua transaksi"?
Kolom rating sengaja dibiarkan memiliki missing value karena rating bersifat opsional—pembeli tidak diwajibkan memberikan rating setelah transaksi selesai. Jika kita mengisi (impute) rating yang kosong dengan nilai tebakan seperti rata-rata atau median, maka hasil perhitungan "rating rata-rata semua transaksi" akan menjadi tidak akurat dan menyesatkan karena mencampurkan data nyata dengan data asumsi. Kepada tim Finance, saya akan menjelaskan bahwa untuk menghitung rating rata-rata yang benar, kita hanya menggunakan transaksi yang memang memiliki rating (mengabaikan nilai NaN), sehingga hasilnya mencerminkan opini pelanggan yang sebenarnya. Pendekatan ini lebih jujur secara statistik dan mencegah distorsi dalam analisis kepuasan pelanggan.